In [ ]:
# -------------------------------
# MedVeriCheck: Prototype Pipeline
# -------------------------------

In [4]:
# 1. Install required packages (run in Colab or local)
!pip install transformers accelerate torch gradio datasets scikit-learn sacremoses > /dev/null 2>&1

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sklearn.metrics import f1_score
from datasets import load_dataset
import re

In [ ]:
# -------------------------------
# 2. Load the lightweight medical model
# Example: "microsoft/BioGPT" or "BioMistral/BioMistral-7B"
# -------------------------------
# MODEL_NAME = "BioMistral/BioMistral-7B"
MODEL_NAME = "microsoft/BioGPT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
# -------------------------------
# 3. Define prompts
# -------------------------------

def baseline_prompt(question):
    return f"Answer the following medical question concisely and accurately:\n\nQuestion: {question}\nAnswer:"

def self_verification_prompt(question, answer):
    return f"""
Can you check your answer if it is right and say a more correct answer.

Question: {question}
Model's answer: {answer}
Verified answer:
"""

In [ ]:
# -------------------------------
# 4. Define generation function
# -------------------------------

def generate_answer(prompt, max_new_tokens=128):
    # output = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    output = pipe(prompt, max_new_tokens=128, do_sample=False, eos_token_id=tokenizer.eos_token_id)

    text = output[0]["generated_text"]
    answer = text[len(prompt):].strip()
    return answer

In [ ]:
dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")

In [ ]:
print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
def get_ground_truth(question, dataset):
    # simple matching: find the closest question
    for item in dataset["train"]:
        if item["Question"].lower() == question.lower():
            return item["Ground Truth"]
    return "Ground truth not found in dataset"

In [ ]:
def answer_question(question):
    # Baseline
    baseline_ans = generate_answer(baseline_prompt(question))

    # Self-verified
    self_verified_ans = generate_answer(self_verification_prompt(question, baseline_ans))

    # Ground truth
    ground_truth = get_ground_truth(question, dataset)

    return {
        "question": question,
        "baseline_answer": baseline_ans,
        "self_verified_answer": self_verified_ans,
        "ground_truth": ground_truth
    }

In [ ]:
def compute_f1(pred, truth):
    # simple word-level F1
    pred_tokens = re.findall(r"\w+", pred.lower())
    truth_tokens = re.findall(r"\w+", truth.lower())

    common = set(pred_tokens) & set(truth_tokens)
    if not pred_tokens or not truth_tokens:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)
    if precision + recall == 0:
        return 0.0
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

In [ ]:
def evaluate_question(question):
    results = answer_question(question)

    baseline_f1 = compute_f1(results["baseline_answer"], results["ground_truth"])
    self_verified_f1 = compute_f1(results["self_verified_answer"], results["ground_truth"])

    print(f"Question: {results['question']}\n")
    print(f"Ground Truth:\n{results['ground_truth']}\n")
    print(f"Baseline Answer:\n{results['baseline_answer']}\nF1 Score: {baseline_f1:.2f}\n")
    print(f"Self-Verified Answer:\n{results['self_verified_answer']}\nF1 Score: {self_verified_f1:.2f}\n")

In [ ]:
my_question = "What is the primary cause of Type 1 Diabetes?"
evaluate_question(my_question)

In [3]:
print("Hello World")

Hello World
